![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 17 -- Lab 1: ECG Beats Close the Gap

Day 16's Lab 2 ended with a gap: the training loss of the plain network went towards zero while the validation loss turned, and the stratified folds scored about 0.87. Today you close that gap one dial at a time: a better optimizer (AdamW), class weights in the loss, dropout, BatchNorm and early stopping. Every step is measured against the same plain run, and at the end the same five folds show how much the recipe is really worth.

**Your role:** Fill in the `# Your code here` cells in order. Run every cell, including the ones already written for you.

**Dataset:** The same ECG beats as Day 16's Lab 2 (Polish AI Olympiad 2025, stage 1, task 3): 2,000 training and 1,500 validation beats of 150 numbers, five classes, 70% normal, scored by balanced accuracy. The setup cell reloads them and brings back the Dataset class, the loaders, `evaluate` and the plain network.

---
# Setup

In [ ]:
# numpy, pandas, matplotlib, scikit-learn and torch are preinstalled on Colab. Uncomment if an import fails.
# !pip install -q numpy pandas matplotlib scikit-learn torch

In [ ]:
import io, copy, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
torch.manual_seed(0)

In [ ]:
URL = "https://raw.githubusercontent.com/OlimpiadaAI/II-OlimpiadaAI/main/1_etap/3_wykrywanie_zaburzen_sygnalu_ekg/train_validation_sets.npz"
d = np.load(io.BytesIO(urllib.request.urlopen(URL).read()))
X_train, y_train = d["X_train"].astype(np.float32), d["y_train"].astype(int)
X_val, y_val = d["X_validation"].astype(np.float32), d["y_validation"].astype(int)
CLASSES = ["normal", "afib", "pac", "pvc", "st_elev"]
counts = np.bincount(y_train)
print(X_train.shape, X_val.shape, counts)

In [ ]:
class BeatDataset(Dataset):                      # Day 16, Lab 2
    def __init__(self, X, y, mean, std):
        self.X = torch.tensor((X - mean) / std, dtype=torch.float32)
        self.y = torch.tensor(y)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        return self.X[i], self.y[i]

mean, std = X_train.mean(axis=0), X_train.std(axis=0)
std[std < 1e-6] = 1.0
train_loader = DataLoader(BeatDataset(X_train, y_train, mean, std), batch_size=64, shuffle=True)
val_loader = DataLoader(BeatDataset(X_val, y_val, mean, std), batch_size=500)

def evaluate(model, loader, loss_fn):
    """Mean loss and balanced accuracy over a loader (eval mode, no slopes)."""
    model.eval()
    total, preds, labels = 0.0, [], []
    with torch.no_grad():
        for xb, yb in loader:
            logits = model(xb)
            total += loss_fn(logits, yb).item() * len(yb)
            preds.append(logits.argmax(1)); labels.append(yb)
    preds, labels = torch.cat(preds).numpy(), torch.cat(labels).numpy()
    return total / len(labels), balanced_accuracy_score(labels, preds)

def make_plain():
    return nn.Sequential(nn.Linear(150, 128), nn.ReLU(), nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 5))

---
# Part 1 -- The plain run, again

The loop of Day 16 with two additions you will need today: it takes an optimizer object and a loss function, and it can stop early (the *Early Stopping: Patience* slide).

## Task 1: The loop with early stopping

Write `fit(model, train_loader, val_loader, opt, loss_fn, epochs, patience=None)`: per epoch the four moves (mean training loss), then `evaluate`; keep `best_score, best_epoch, best_state` by validation **balanced accuracy**; if `patience` is given and `epoch - best_epoch >= patience`, stop. Return `history` (list of `(train_loss, val_loss, val_bal)`) and `(best_score, best_epoch, best_state)`. Then run the plain baseline: `torch.manual_seed(0); model = make_plain()`, `opt = torch.optim.SGD(model.parameters(), lr=0.05)`, `nn.CrossEntropyLoss()`, 60 epochs, no patience. Store the result as `hist_plain, best_plain` and print the best score and epoch, the final training loss and the final validation loss. Expected: best about 0.94 late in the run, a training loss near 0.01, and a validation loss stuck at about 0.15 while the training loss keeps falling.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert len(hist_plain) == 60 and best_plain[0] > 0.85, 'fit: SGD lr 0.05 for 60 epochs, best state by validation balanced accuracy'
print('Task 1 passed')

---
# Part 2 -- One dial at a time

Each task changes exactly one thing and keeps the seed, so the difference is the dial and nothing else (the *common mistake* of the Limitations slide).

## Task 2: AdamW instead of SGD

Same plain network (`torch.manual_seed(0); model = make_plain()`), same loss, but `opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)`, 60 epochs. Store `hist_adamw, best_adamw` and report. Then print the validation balanced accuracy of the first eight epochs of both runs (`[round(h[2], 3) for h in hist_plain[:8]]` and the same for `hist_adamw`): which run leaves the 0.2 floor first? Look also at the final training loss and the final validation loss. What did AdamW fix, and what did it not fix?

In [ ]:
# Your code here


## Task 3: Class weights in the loss

The *Weights for Rare Classes* slide of Day 16. Build `w = torch.tensor(counts.sum() / (5 * counts), dtype=torch.float32)` (print it: the normal class gets about 0.29, each disorder about 2.67) and `loss_w = nn.CrossEntropyLoss(weight=w)`. Train the plain network with AdamW and `loss_w` for 60 epochs as `hist_w, best_w`. Does the best balanced accuracy move, and at which epoch is it reached now?

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert abs(w[0].item() - 0.2857) < 0.01 and abs(w[1].item() - 2.667) < 0.01, 'weights: counts.sum() / (5 * counts): 2000 / (5 * 1400) = 0.286 for the normal class'
print('Task 3 passed')

## Task 4: Dropout

Write `make_dropout(p=0.3)` returning the plain network with `nn.Dropout(p)` after each `nn.ReLU()`. Train it with AdamW and `loss_w` for 60 epochs as `hist_drop, best_drop`. Compare the final training loss with Task 3's: the holes keep the network from memorising, so the training loss stays higher while the validation score does not get worse.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert sum(isinstance(m, nn.Dropout) for m in make_dropout()) == 2, 'make_dropout: one nn.Dropout(p) after each ReLU'
print('Task 4 passed')

## Task 5: BatchNorm

Write `make_recipe(p=0.3)`: `nn.Linear(150, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(p), nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(p), nn.Linear(64, 5)`. Train it with AdamW and `loss_w` for 60 epochs as `hist_bn, best_bn`. Expected: about 0.97 again, with the lowest validation loss of all the runs. BatchNorm's big gains are on deep networks (the deck's five-layer run); here it mostly steadies the training.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert best_bn[0] > best_plain[0], 'recipe: BatchNorm after each Linear, dropout after each ReLU, trained with AdamW and the weighted loss'
print('Task 5 passed')

## Task 6: Early stopping

Train the recipe once more with `patience=10` (and up to 100 epochs) as `hist_es, best_es`. Print at which epoch it stopped (`len(hist_es)`) and the best epoch it kept. Save the kept dials with `torch.save(best_es[2], "ecg_recipe.pt")`.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
import os
assert len(hist_es) < 100 and os.path.exists('ecg_recipe.pt'), 'early stopping: patience 10 must stop before epoch 100; save best_es[2]'
print('Task 6 passed')

---
# Part 3 -- What each dial bought

## Task 7: The ladder

Build a `pd.DataFrame` with one row per run (plain, + AdamW, + class weights, + dropout, + BatchNorm, + early stopping) and the columns `best`, `best_epoch`, `final_train_loss`, `final_val_loss` taken from the histories and bests. Print it. Which dial bought the most?

In [ ]:
# Your code here


## Task 8: Plain against recipe, on one figure

Left: the training and validation losses of the plain run (solid) and of the recipe run of Task 5 (dashed). Right: the validation balanced accuracy of both. Describe the two gaps in one sentence each.

In [ ]:
# Your code here


---
# Part 4 -- The folds decide

Day 16's Lab 2 ran the plain network on five stratified folds. The same folds, the same seed, the recipe.

## Task 9: Stratified 5-fold with the recipe

Repeat Day 16's fold loop (`StratifiedKFold(5, shuffle=True, random_state=0)`, standardisation from each fold's training part, `torch.manual_seed(0)` before each model) with `make_recipe()`, AdamW, `loss_w`, up to 40 epochs and `patience=10`. Store the fold scores in `fold_recipe` and print their mean and spread. Then rerun the plain network on the same folds (SGD lr 0.05, 40 epochs, no patience) into `fold_plain` and print both. Expected: about 0.95 against about 0.87, with the recipe better on every fold.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert np.mean(fold_recipe) > np.mean(fold_plain) + 0.03, 'folds: the recipe must beat the plain network by a clear margin on the same five folds'
print('Task 9 passed')

## Task 10: The confusion matrices

Load `ecg_recipe.pt` into a fresh `make_recipe()`, collect its predictions on `val_loader` (eval mode, `no_grad`) and print `confusion_matrix(y_val, preds)`; do the same for the plain run's `best_plain[2]` in a fresh `make_plain()`. Which class gained the most recall?

In [ ]:
# Your code here


---
## Task 11: Reflect

AdamW alone drove the training loss to almost zero faster than SGD did. In one sentence: what does an optimizer fix, and what does it not? Dropout made the training loss *worse* and the validation score no worse. Why is that the sign it is working? The folds say about 0.95 for the recipe and the official validation set says about 0.97. Which number do you report, and what would you tell a teammate who wants to tune the dropout rate on the official validation set?

*Your answers here*

---
## Conclusion

- **One dial at a time**: the same seed, the same folds, one change per run; the ladder shows what each dial bought and what it did not.
- **AdamW** shortened the stalled start and removed the rate search; **class weights** made the rare disorders count; **dropout** closed the gap between the training and validation losses and bought the most; **BatchNorm** steadied the run; **early stopping** found the epoch without guessing.
- **The recipe** lifted the stratified 5-fold score from about 0.87 to about 0.95, and the official validation balanced accuracy from 0.94 to about 0.97, on the same raw 150 numbers.
- **The dashboard**: a training loss that stays near the validation loss is the picture of a network that generalises.

---

Made By **Sattam Altwaim**